# Thesis visuals & DM analysis (robust notebook)
Put this file into your project root (e.g., `D:\\progr`) and **Run All**.

## What this notebook does
1. Loads `outputs/results/metrics.csv` and `outputs/results/dm_tests.csv` (if present)
2. Scans `outputs/predictions/*.csv` (pipeline exports)
3. Trains **XGBoost on GPU** (RTX 4060) per file and compares vs **DCC_GARCH** using **Diebold–Mariano test**
4. Saves figures to `outputs/figures_notebook/` and LaTeX tables to `outputs/tables_notebook/`

## Required before running
**First run your pipeline** so the outputs exist:
```powershell
D:\\progr\.venv\\Scripts\\python.exe D:\\progr\\main.py
```


In [1]:

# === 0) Imports (NO seaborn) ===
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Optional: GPU XGBoost
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost not available:", e)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:

# === 1) Paths ===
ROOT = Path(".").resolve()
OUTPUTS = ROOT / "outputs"
RESULTS_DIR = OUTPUTS / "results"
PRED_DIR = OUTPUTS / "predictions"

FIG_DIR = OUTPUTS / "figures_notebook"
TAB_DIR = OUTPUTS / "tables_notebook"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook ROOT:", ROOT)
print("Pred dir:", PRED_DIR, "| exists:", PRED_DIR.exists())


Notebook ROOT: D:\progr
Pred dir: D:\progr\outputs\predictions | exists: True


In [3]:

# === 2) Load metrics + DM tests (if present) ===
metrics_path = RESULTS_DIR / "metrics.csv"
dm_path = RESULTS_DIR / "dm_tests.csv"

assert metrics_path.exists(), f"Missing {metrics_path}. Run your pipeline first."
metrics = pd.read_csv(metrics_path)

# Normalize schema
if "w" in metrics.columns and "window" not in metrics.columns:
    metrics = metrics.rename(columns={"w":"window"})
if "space" in metrics.columns and "target_space" not in metrics.columns:
    metrics = metrics.rename(columns={"space":"target_space"})
if "n" in metrics.columns and "n_test" not in metrics.columns:
    metrics = metrics.rename(columns={"n":"n_test"})

required = {"dependency","window","model","RMSE"}
missing = required - set(metrics.columns)
assert not missing, f"metrics.csv missing columns: {missing}. Found: {list(metrics.columns)}"

print("metrics shape:", metrics.shape)
display(metrics.head(10))

if dm_path.exists():
    dm = pd.read_csv(dm_path)
    if "w" in dm.columns and "window" not in dm.columns:
        dm = dm.rename(columns={"w":"window"})
    if "target_space" in dm.columns and "space" not in dm.columns:
        dm = dm.rename(columns={"target_space":"space"})
    if "dm_stat" in dm.columns and "DM_stat" not in dm.columns:
        dm = dm.rename(columns={"dm_stat":"DM_stat"})
    print("dm_tests shape:", dm.shape)
    display(dm.head(10))
else:
    dm = None
    print("dm_tests.csv not found (OK).")


metrics shape: (120, 8)


,dependency,window,target_space,model,MAE,RMSE,R2,n_test
0,corr_BTC-USD_^GSPC,14,fisher_z,AR1,0.088039,0.126297,0.868947,1255
1,corr_BTC-USD_^GSPC,14,fisher_z,Naive_Last,0.087919,0.127862,0.865678,1255
2,corr_BTC-USD_^GSPC,14,fisher_z,Ridge,0.132550,0.175257,0.747644,1255
3,corr_BTC-USD_^GSPC,14,fisher_z,RF,0.144552,0.186337,0.714725,1255
4,corr_BTC-USD_^GSPC,14,fisher_z,ElasticNet,0.320505,0.396812,-0.293702,1255
5,corr_BTC-USD_^GSPC,30,fisher_z,AR1,0.037334,0.058634,0.950826,1239
6,corr_BTC-USD_^GSPC,30,fisher_z,Naive_Last,0.036935,0.058815,0.950522,1239
7,corr_BTC-USD_^GSPC,30,fisher_z,Ridge,0.057792,0.083880,0.899365,1239
8,corr_BTC-USD_^GSPC,30,fisher_z,RF,0.067173,0.093549,0.874826,1239
9,corr_BTC-USD_^GSPC,30,fisher_z,ElasticNet,0.269685,0.323107,-0.493237,1239


dm_tests shape: (48, 8)


,dependency,window,space,model,benchmark,DM_stat,p_value,n
0,corr_BTC-USD_^GSPC,14,fisher_z,Ridge,Naive_Last,-11.880939,0.000000e+00,1255.0
1,corr_BTC-USD_^GSPC,14,fisher_z,Ridge,DCC_GARCH,16.354195,0.000000e+00,1255.0
2,corr_BTC-USD_^GSPC,30,fisher_z,Ridge,Naive_Last,-9.833022,0.000000e+00,1239.0
3,corr_BTC-USD_^GSPC,30,fisher_z,Ridge,DCC_GARCH,18.051013,0.000000e+00,1239.0
4,corr_BTC-USD_^GSPC,60,fisher_z,Ridge,Naive_Last,-8.205213,6.661338e-16,1209.0
5,corr_BTC-USD_^GSPC,60,fisher_z,Ridge,DCC_GARCH,22.059640,0.000000e+00,1209.0
6,corr_BTC-USD_^GSPC,90,fisher_z,Ridge,Naive_Last,-7.045262,3.141931e-12,1179.0
7,corr_BTC-USD_^GSPC,90,fisher_z,Ridge,DCC_GARCH,27.272930,0.000000e+00,1179.0
8,corr_BTC-USD_^IXIC,14,fisher_z,Ridge,Naive_Last,-11.722002,0.000000e+00,1255.0
9,corr_BTC-USD_^IXIC,14,fisher_z,Ridge,DCC_GARCH,15.910565,0.000000e+00,1255.0


In [4]:

# === 3) Best model table + RMSE vs window plots ===
best = (metrics.sort_values(["dependency","window","RMSE"])
        .groupby(["dependency","window"], as_index=False)
        .first())
display(best.head(20))

def safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+","_", s)

for dep in sorted(metrics["dependency"].unique()):
    df = metrics[metrics["dependency"] == dep].sort_values(["model","window"])
    fig, ax = plt.subplots(figsize=(8,4.5))
    for model, g in df.groupby("model"):
        ax.plot(g["window"], g["RMSE"], marker="o", label=model)
    ax.set_title(f"RMSE vs window | {dep}")
    ax.set_xlabel("Rolling window (days)")
    ax.set_ylabel("RMSE (Fisher-z unless specified)")
    ax.grid(True, alpha=0.3)
    ax.legend(ncols=2, fontsize=8)
    out = FIG_DIR / f"rmse_vs_window_{safe_name(dep)}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    plt.close(fig)

print("Saved plots ->", FIG_DIR)


,dependency,window,target_space,model,MAE,RMSE,R2,n_test
0,corr_BTC-USD_ETH-USD,14,fisher_z,AR1,0.101089,0.156772,0.875030,1255
1,corr_BTC-USD_ETH-USD,30,fisher_z,AR1,0.043828,0.071976,0.956765,1239
2,corr_BTC-USD_ETH-USD,60,fisher_z,AR1,0.020726,0.037388,0.980280,1209
3,corr_BTC-USD_ETH-USD,90,fisher_z,AR1,0.012824,0.023703,0.987883,1179
4,corr_BTC-USD_GLD,14,fisher_z,AR1,0.088535,0.135544,0.838662,1255
5,corr_BTC-USD_GLD,30,fisher_z,AR1,0.037582,0.062366,0.927341,1239
6,corr_BTC-USD_GLD,60,fisher_z,AR1,0.018335,0.033244,0.963336,1209
7,corr_BTC-USD_GLD,90,fisher_z,AR1,0.011824,0.021250,0.979735,1179
8,corr_BTC-USD_SLV,14,fisher_z,AR1,0.083070,0.123112,0.860062,1255
9,corr_BTC-USD_SLV,30,fisher_z,AR1,0.036637,0.058208,0.941800,1239


Saved plots -> D:\progr\outputs\figures_notebook


In [5]:

# === 4) Export LaTeX tables (escape underscores) ===
def df_to_latex(df: pd.DataFrame, caption: str, label: str) -> str:
    safe = df.copy()
    for c in safe.columns:
        if safe[c].dtype == object:
            safe[c] = safe[c].astype(str).str.replace("_", r"\_", regex=False)
    return safe.to_latex(index=False, escape=False, caption=caption, label=label)

metrics_tex = df_to_latex(metrics, "Out-of-sample forecasting metrics.", "tab:metrics_full")
best_tex = df_to_latex(best[["dependency","window","model","RMSE"]], "Best model per dependency/window by RMSE.", "tab:best_models")

(TAB_DIR / "metrics_full.tex").write_text(metrics_tex, encoding="utf-8")
(TAB_DIR / "best_models.tex").write_text(best_tex, encoding="utf-8")
print("Saved LaTeX tables ->", TAB_DIR)


Saved LaTeX tables -> D:\progr\outputs\tables_notebook


In [6]:

# === 5) DM test implementation ===
from math import sqrt
try:
    from scipy.stats import t as student_t
    HAS_SCIPY = True
except Exception as e:
    HAS_SCIPY = False
    print("SciPy not available; DM tests will be skipped:", e)

def dm_test(e_model: np.ndarray, e_bench: np.ndarray, h: int = 1):
    # d_t = e_model^2 - e_bench^2, so DM<0 => model better
    e_model = np.asarray(e_model, dtype=float)
    e_bench = np.asarray(e_bench, dtype=float)
    mask = np.isfinite(e_model) & np.isfinite(e_bench)
    e_model = e_model[mask]
    e_bench = e_bench[mask]
    T = len(e_model)
    if (T < 5) or (not HAS_SCIPY):
        return np.nan, np.nan, T
    d = (e_model**2) - (e_bench**2)
    lag = max(h-1, 0)
    dbar = d.mean()
    gamma0 = np.mean((d-dbar)*(d-dbar))
    var = gamma0
    for k in range(1, lag+1):
        gamma = np.mean((d[k:]-dbar)*(d[:-k]-dbar))
        var += 2*(1-k/(lag+1))*gamma
    var = var / T
    if var <= 0 or not np.isfinite(var):
        return np.nan, np.nan, T
    dm_stat = dbar / sqrt(var)
    p = 2*(1-student_t.cdf(abs(dm_stat), df=T-1))
    return float(dm_stat), float(p), int(T)


## 6) XGBoost GPU vs DCC_GARCH (automatic loop)
Scans `outputs/predictions/*_w*_fisher_z_predictions.csv`. For each file with `DCC_GARCH`,
trains XGB on GPU from lag features of `y_true` and computes DM test **XGB vs DCC**.

**DM sign convention:** we compute $d_t=e_{XGB}^2-e_{DCC}^2$ → **DM<0 means XGB is better**.

In [7]:

# === 6) Scan prediction exports ===
assert PRED_DIR.exists(), f"Missing predictions dir: {PRED_DIR}. Run pipeline first."
pred_files = sorted(PRED_DIR.glob("*_w*_fisher_z_predictions.csv"))
print("Found prediction files:", len(pred_files))
for f in pred_files[:15]:
    print(" -", f.name)
if len(pred_files) == 0:
    raise RuntimeError("No *_w*_fisher_z_predictions.csv found.")


Found prediction files: 24
 - corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w60_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w90_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w14_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w30_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w60_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w90_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w30_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w60_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w90_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w14_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w30_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w60_fisher_z_predictions.csv


In [8]:

# === 7) Helpers: parse + features ===
def parse_dep_window(fname: str):
    m = re.match(r"(.+)_w(\d+)_fisher_z_predictions\.csv$", fname)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

def build_lag_features(y: np.ndarray, lags: int):
    y = np.asarray(y, dtype=float)
    X, yy, idx = [], [], []
    for t in range(lags, len(y)):
        window = y[t-lags:t]
        if np.any(~np.isfinite(window)) or (not np.isfinite(y[t])):
            continue
        X.append(window[::-1])
        yy.append(y[t])
        idx.append(t)
    if len(yy) == 0:
        return np.empty((0,lags)), np.empty((0,)), np.empty((0,), dtype=int)
    return np.vstack(X), np.array(yy), np.array(idx, dtype=int)

def split_time(X, y, idx, test_ratio=0.2):
    n = len(y)
    n_test = int(round(n*test_ratio))
    n_test = max(n_test, 50) if n >= 250 else max(n_test, 10)
    n_test = min(n_test, n-1)
    n_train = n - n_test
    return X[:n_train], X[n_train:], y[:n_train], y[n_train:], idx[:n_train], idx[n_train:]


In [9]:

# === 8) Run loop ===
rows = []
if not HAS_XGB:
    print("XGBoost not installed. Install xgboost with CUDA support to run this part.")
else:
    for f in pred_files:
        dep, w = parse_dep_window(f.name)
        if dep is None:
            continue

        df = pd.read_csv(f)
        if "DCC_GARCH" not in df.columns:
            continue
        if "y_true" not in df.columns:
            continue

        y = df["y_true"].astype(float).values
        dcc = df["DCC_GARCH"].astype(float).values

        LAGS = 30
        X, yy, idx = build_lag_features(y, lags=LAGS)

        # Align DCC with built rows
        dcc_aligned = dcc[idx]

        # Drop NaNs in DCC
        mask = np.isfinite(dcc_aligned) & np.isfinite(yy)
        X, yy, idx, dcc_aligned = X[mask], yy[mask], idx[mask], dcc_aligned[mask]

        if len(yy) < 200:
            continue

        X_tr, X_te, y_tr, y_te, idx_tr, idx_te = split_time(X, yy, idx, test_ratio=0.2)
        dcc_te = dcc_aligned[len(y_tr):]

        params = dict(
            n_estimators=800,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="reg:squarederror",
            tree_method="hist",
            device="cuda",
            random_state=42,
        )
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr)

        pred_xgb = model.predict(X_te)
        pred_dcc = dcc_te

        rmse_xgb = float(np.sqrt(mean_squared_error(y_te, pred_xgb)))
        rmse_dcc = float(np.sqrt(mean_squared_error(y_te, pred_dcc)))

        dm_stat, pval, n = dm_test(y_te - pred_xgb, y_te - pred_dcc, h=1)

        rows.append({
            "dependency": dep,
            "window": w,
            "DM_stat": dm_stat,
            "p_value": pval,
            "n": n,
            "RMSE_XGB": rmse_xgb,
            "RMSE_DCC": rmse_dcc,
            "file": f.name
        })

dm_xgb_vs_dcc = pd.DataFrame(rows)
print("Rows computed:", len(dm_xgb_vs_dcc))
display(dm_xgb_vs_dcc.head(20))

out_csv = RESULTS_DIR / "dm_xgb_vs_dcc.csv"
dm_xgb_vs_dcc.to_csv(out_csv, index=False)
print("Saved ->", out_csv)


D:\progr\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [22:21:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Rows computed: 24


,dependency,window,DM_stat,p_value,n,RMSE_XGB,RMSE_DCC,file
0,corr_BTC-USD_^GSPC,14,-11.888241,0.0,405,0.150763,0.313846,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv
1,corr_BTC-USD_^GSPC,30,-11.134188,0.0,402,0.069714,0.179033,corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv
2,corr_BTC-USD_^GSPC,60,-15.190753,0.0,396,0.039372,0.138831,corr_BTC-USD_^GSPC_w60_fisher_z_predictions.csv
3,corr_BTC-USD_^GSPC,90,-20.735725,0.0,390,0.026746,0.116303,corr_BTC-USD_^GSPC_w90_fisher_z_predictions.csv
4,corr_BTC-USD_^IXIC,14,-11.377163,0.0,405,0.152362,0.313793,corr_BTC-USD_^IXIC_w14_fisher_z_predictions.csv
5,corr_BTC-USD_^IXIC,30,-11.953133,0.0,402,0.067910,0.170093,corr_BTC-USD_^IXIC_w30_fisher_z_predictions.csv
6,corr_BTC-USD_^IXIC,60,-16.238678,0.0,396,0.036707,0.120879,corr_BTC-USD_^IXIC_w60_fisher_z_predictions.csv
7,corr_BTC-USD_^IXIC,90,-19.934869,0.0,390,0.023788,0.101095,corr_BTC-USD_^IXIC_w90_fisher_z_predictions.csv
8,corr_BTC-USD_ETH-USD,14,-11.936744,0.0,405,0.147900,0.295663,corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv
9,corr_BTC-USD_ETH-USD,30,-13.961786,0.0,402,0.072483,0.178799,corr_BTC-USD_ETH-USD_w30_fisher_z_predictions.csv


Saved -> D:\progr\outputs\results\dm_xgb_vs_dcc.csv


In [10]:

# === 9) Plot XGB vs DCC RMSE per dependency ===
def safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+","_", s)

if len(dm_xgb_vs_dcc) == 0:
    print("No rows to plot (check that prediction files contain DCC_GARCH and y_true).")
else:
    for dep, g in dm_xgb_vs_dcc.groupby("dependency"):
        g = g.sort_values("window")
        x = np.arange(len(g))
        width = 0.38
        fig, ax = plt.subplots(figsize=(9,4.5))
        ax.bar(x - width/2, g["RMSE_XGB"], width, label="XGB (GPU retrained)")
        ax.bar(x + width/2, g["RMSE_DCC"], width, label="DCC_GARCH")
        ax.set_xticks(x)
        ax.set_xticklabels(g["window"].astype(int).astype(str))
        ax.set_title(f"RMSE | XGB vs DCC | {dep}")
        ax.set_xlabel("Window (days)")
        ax.set_ylabel("RMSE (Fisher-z)")
        ax.grid(True, alpha=0.3, axis="y")
        ax.legend()
        out = FIG_DIR / f"rmse_xgb_vs_dcc_{safe_name(dep)}.png"
        fig.tight_layout()
        fig.savefig(out, dpi=200)
        plt.close(fig)

    # Save LaTeX table
    dm_tex = df_to_latex(dm_xgb_vs_dcc[["dependency","window","DM_stat","p_value","RMSE_XGB","RMSE_DCC","n"]],
                         "Diebold--Mariano test: XGB (GPU retrained) vs DCC\\_GARCH. We use $d_t=e_{XGB}^2-e_{DCC}^2$; DM<0 means XGB is better.",
                         "tab:dm_xgb_vs_dcc")
    (TAB_DIR / "dm_xgb_vs_dcc.tex").write_text(dm_tex, encoding="utf-8")

    print("Saved plots ->", FIG_DIR)
    print("Saved LaTeX ->", TAB_DIR / "dm_xgb_vs_dcc.tex")


Saved plots -> D:\progr\outputs\figures_notebook
Saved LaTeX -> D:\progr\outputs\tables_notebook\dm_xgb_vs_dcc.tex


## If something breaks
- **`NameError: RESULTS`** → you started from the middle; run from the top (Run All)
- **XGBoost device warning** → OK; this notebook uses `device='cuda'` and `tree_method='hist'` for GPU.
- **`Found array with 0 sample(s)`** → prediction file had no usable rows after NaN filtering; the notebook skips such files.
